# Advanced Pull Pipelines — A Step-by-Step Tutorial with Problems and Solutions

This notebook continues the topic of **pulling data through generator pipelines**.

The style is deliberately incremental:

- introduce one idea;
- inspect a small result;
- add one transformation;
- connect the stages;
- explain what is being pulled and when;
- solve a larger problem using the smaller pieces.

The notebook is self-contained. It creates a semicolon-delimited car dataset with the same header style as the course file:

```text
Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
```

## What makes these problems advanced?

We will move beyond a single filter and study:

- schema-aware row projection;
- higher-order generator stages;
- early termination;
- typed parsing;
- validation with a side channel;
- stateful streaming transformations;
- duplicate removal;
- rolling calculations;
- bounded-memory batching;
- one-pass aggregation;
- top-*k* selection per group;
- replayable sources;
- configuration-driven pipelines;
- explicit backpressure;
- sinks and audit reports;
- an end-to-end capstone pipeline.

The emphasis is not just on getting the right answer. We will also ask:

> How much data was read? What state was stored? Can the stage be reused? What happens when input is malformed?

## 0. Imports and sample data

We start with imports and create a local data file. The notebook always writes its own file so that every example is reproducible.

In [3]:
from pathlib import Path
from dataclasses import dataclass, replace, asdict
from collections import defaultdict, deque
from typing import Any, Callable, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, TypeVar
import csv
import heapq
import itertools
import math
import operator
import statistics
import tempfile

DATA_FILE = Path("cars_tutorial.csv")

DATA = """Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
Chevrolet Chevelle Malibu;18.0;8;307.0;130.0;3504.;12.0;70;US
Buick Skylark 320;15.0;8;350.0;165.0;3693.;11.5;70;US
Plymouth Satellite;18.0;8;318.0;150.0;3436.;11.0;70;US
AMC Rebel SST;16.0;8;304.0;150.0;3433.;12.0;70;US
Ford Torino;17.0;8;302.0;140.0;3449.;10.5;70;US
Ford Galaxie 500;15.0;8;429.0;198.0;4341.;10.0;70;US
Chevrolet Impala;14.0;8;454.0;220.0;4354.;9.0;70;US
Chevrolet Monte Carlo;15.0;8;400.0;150.0;3761.;9.5;70;US
Toyota Corolla;31.0;4;71.0;65.0;1773.;19.0;71;Japan
Datsun 510;27.0;4;97.0;88.0;2130.;14.5;71;Japan
Volkswagen 1131 Deluxe Sedan;26.0;4;97.0;46.0;1835.;20.5;70;Europe
Peugeot 504;25.0;4;110.0;87.0;2672.;17.5;70;Europe
Audi 100 LS;24.0;4;107.0;90.0;2430.;14.5;70;Europe
Saab 99e;25.0;4;104.0;95.0;2375.;17.5;70;Europe
BMW 2002;26.0;4;121.0;113.0;2234.;12.5;70;Europe
Chevrolet Vega 2300;28.0;4;140.0;90.0;2264.;15.5;71;US
Ford Pinto;25.0;4;98.0;;2046.;19.0;71;US
AMC Gremlin;21.0;6;199.0;90.0;2648.;15.0;70;US
Mazda RX-3;18.0;3;70.0;90.0;2124.;13.5;73;Japan
Renault 12;26.0;4;96.0;69.0;2189.;18.0;72;Europe
Dodge Colt;28.0;4;90.0;75.0;2125.;14.5;74;US
Volvo 244DL;22.0;4;121.0;98.0;2945.;14.5;75;Europe
Honda Civic;33.0;4;91.0;53.0;1795.;17.4;76;Japan
Chevrolet Monte Carlo S;15.0;8;350.0;145.0;4082.;13.0;73;US
Chevrolet Monte Carlo Landau;15.5;8;350.0;170.0;4165.;11.4;77;US
Honda Accord LX;29.5;4;98.0;68.0;2135.;16.6;78;Japan
Chevrolet Monte Carlo Landau;19.2;8;305.0;145.0;3425.;13.2;78;US
Mercedes-Benz 240d;30.0;4;146.0;67.0;3250.;21.8;80;Europe
Subaru GL;33.8;4;97.0;67.0;2145.;18.0;80;Japan
Toyota Celica GT;32.0;4;144.0;96.0;2665.;13.9;82;Japan
Honda Civic;33.0;4;91.0;53.0;1795.;17.4;76;Japan
"""

# DATA_FILE.write_text(DATA, encoding="utf-8")
# print(DATA_FILE.resolve())

The final row intentionally duplicates an earlier Honda Civic row. We will use it later when we study stateful duplicate removal.

# Problem 1 — Build a schema-aware pull source

The original style starts with a generator that pulls rows from a CSV file.

This time, we want each row to be a dictionary rather than a list. But we do not want downstream code to depend on course-specific headers such as `Car` and `Model`.

We will normalize the headers once at the source boundary.

### Step 1: define the canonical column names

The rest of the notebook will use:

```text
name, mpg, cylinders, displacement, horsepower,
weight, acceleration, model_year, origin
```

In [4]:
HEADER_MAP = {
    "Car": "name",
    "MPG": "mpg",
    "Cylinders": "cylinders",
    "Displacement": "displacement",
    "Horsepower": "horsepower",
    "Weight": "weight",
    "Acceleration": "acceleration",
    "Model": "model_year",
    "Origin": "origin",
}

### Step 2: write the source generator

The generator opens the file only when iteration begins. It closes the file automatically when iteration finishes.

In [5]:
def pull_rows(file_name: Path) -> Iterator[Dict[str, str]]:
    with file_name.open("r", encoding="utf-8", newline="") as file_obj:
        reader = csv.DictReader(file_obj, delimiter=";")

        if reader.fieldnames is None:
            raise ValueError("The CSV file does not contain a header row.")

        missing_headers = set(HEADER_MAP) - set(reader.fieldnames)
        if missing_headers:
            raise ValueError(
                "Missing required headers: " + ", ".join(sorted(missing_headers))
            )

        for raw_row in reader:
            yield {
                canonical_name: raw_row[source_name]
                for source_name, canonical_name in HEADER_MAP.items()
            }

### Step 3: pull only three rows

`itertools.islice` does not load the complete file. It asks the source for three values and then stops.

In [6]:
for row in itertools.islice(pull_rows(DATA_FILE), 3):
    print(row)

{'name': 'Chevrolet Chevelle Malibu', 'mpg': '18.0', 'cylinders': '8', 'displacement': '307.0', 'horsepower': '130.0', 'weight': '3504.', 'acceleration': '12.0', 'model_year': '70', 'origin': 'US'}
{'name': 'Buick Skylark 320', 'mpg': '15.0', 'cylinders': '8', 'displacement': '350.0', 'horsepower': '165.0', 'weight': '3693.', 'acceleration': '11.5', 'model_year': '70', 'origin': 'US'}
{'name': 'Plymouth Satellite', 'mpg': '18.0', 'cylinders': '8', 'displacement': '318.0', 'horsepower': '150.0', 'weight': '3436.', 'acceleration': '11.0', 'model_year': '70', 'origin': 'US'}


The source has now established a stable schema. Every later generator can ignore the original capitalization and delimiter details.

# Problem 2 — Project only the columns a consumer needs

A common pipeline mistake is to carry large records through every stage even when the consumer needs only a few fields.

Write a generator that selects a subset of dictionary keys.

### Step 1: implement a projection stage

The stage receives rows from upstream and yields smaller dictionaries downstream.

In [7]:
def select_fields(
    rows: Iterable[Dict[str, Any]],
    *field_names: str,
) -> Iterator[Dict[str, Any]]:
    for row in rows:
        yield {field_name: row[field_name] for field_name in field_names}

### Step 2: connect the source to the projection

The pipeline is:

```text
caller <-- select_fields <-- pull_rows <-- file
```

In [8]:
projected = select_fields(
    pull_rows(DATA_FILE),
    "name",
    "mpg",
    "origin",
)

for row in itertools.islice(projected, 5):
    print(row)

{'name': 'Chevrolet Chevelle Malibu', 'mpg': '18.0', 'origin': 'US'}
{'name': 'Buick Skylark 320', 'mpg': '15.0', 'origin': 'US'}
{'name': 'Plymouth Satellite', 'mpg': '18.0', 'origin': 'US'}
{'name': 'AMC Rebel SST', 'mpg': '16.0', 'origin': 'US'}
{'name': 'Ford Torino', 'mpg': '17.0', 'origin': 'US'}


Projection is lazy. The dictionaries are created one at a time, only when the caller asks for them.

# Problem 3 — Build higher-order filtering stages

Instead of writing one filter function for every condition, create a generic generator that accepts a predicate.

A predicate is simply a function that returns `True` or `False` for one item.

### Step 1: implement the generic filter

In [9]:
T = TypeVar("T")


def where(items: Iterable[T], predicate: Callable[[T], bool]) -> Iterator[T]:
    for item in items:
        if predicate(item):
            yield item

### Step 2: create a predicate factory for text matching

The factory returns a new predicate configured with the requested words.

In [10]:
def name_contains_all(*words: str) -> Callable[[Dict[str, str]], bool]:
    normalized_words = [word.casefold() for word in words]

    def predicate(row: Dict[str, str]) -> bool:
        candidate = row["name"].casefold()
        return all(word in candidate for word in normalized_words)

    return predicate

### Step 3: find Chevrolet Monte Carlo records

In [11]:
monte_carlos = where(
    pull_rows(DATA_FILE),
    name_contains_all("Chevrolet", "Carlo"),
)

for row in monte_carlos:
    print(row["name"], row["mpg"], row["model_year"])

Chevrolet Monte Carlo 15.0 70
Chevrolet Monte Carlo S 15.0 73
Chevrolet Monte Carlo Landau 15.5 77
Chevrolet Monte Carlo Landau 19.2 78


The filtering stage does not know anything about cars. It only knows how to call a predicate. This separation makes the stage reusable.

# Problem 4 — Convert text rows into typed records

CSV readers produce strings. Numeric comparison on strings is dangerous:

```python
"9" > "30"   # True, because strings are compared lexicographically
```

We therefore convert rows near the beginning of the pipeline.

### Step 1: define a typed record

In [12]:
@dataclass(frozen=True)
class CarRecord:
    name: str
    mpg: Optional[float]
    cylinders: int
    displacement: float
    horsepower: Optional[float]
    weight: float
    acceleration: float
    model_year: int
    origin: str

### Step 2: define small conversion helpers

Missing MPG or horsepower values are represented by `None`.

In [13]:
def optional_float(value: str) -> Optional[float]:
    value = value.strip()
    if value in {"", "?", "NA"}:
        return None
    return float(value)


def parse_record(row: Dict[str, str]) -> CarRecord:
    return CarRecord(
        name=row["name"].strip(),
        mpg=optional_float(row["mpg"]),
        cylinders=int(row["cylinders"]),
        displacement=float(row["displacement"]),
        horsepower=optional_float(row["horsepower"]),
        weight=float(row["weight"]),
        acceleration=float(row["acceleration"]),
        model_year=int(row["model_year"]),
        origin=row["origin"].strip(),
    )

### Step 3: create a mapping generator

In [14]:
U = TypeVar("U")


def transform(
    items: Iterable[T],
    function: Callable[[T], U],
) -> Iterator[U]:
    for item in items:
        yield function(item)

### Step 4: connect parsing to the source

The pipeline is now:

```text
caller <-- parse_record <-- pull_rows <-- file
```

In [15]:
def pull_cars(file_name: Path) -> Iterator[CarRecord]:
    yield from transform(pull_rows(file_name), parse_record)


for car in itertools.islice(pull_cars(DATA_FILE), 4):
    print(car)

CarRecord(name='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, displacement=307.0, horsepower=130.0, weight=3504.0, acceleration=12.0, model_year=70, origin='US')
CarRecord(name='Buick Skylark 320', mpg=15.0, cylinders=8, displacement=350.0, horsepower=165.0, weight=3693.0, acceleration=11.5, model_year=70, origin='US')
CarRecord(name='Plymouth Satellite', mpg=18.0, cylinders=8, displacement=318.0, horsepower=150.0, weight=3436.0, acceleration=11.0, model_year=70, origin='US')
CarRecord(name='AMC Rebel SST', mpg=16.0, cylinders=8, displacement=304.0, horsepower=150.0, weight=3433.0, acceleration=12.0, model_year=70, origin='US')


`yield from` delegates iteration to the transformation generator. The consumer still pulls one typed record at a time.

# Problem 5 — Stop a pipeline at a data-dependent boundary

`islice` stops after a fixed number of items. Sometimes the stopping rule depends on the data.

Write a `take_until` generator that stops when a predicate becomes true.

### Step 1: decide whether the boundary item is included

We will make this behavior configurable.

In [16]:
def take_until(
    items: Iterable[T],
    stop_predicate: Callable[[T], bool],
    include_boundary: bool = False,
) -> Iterator[T]:
    for item in items:
        if stop_predicate(item):
            if include_boundary:
                yield item
            return
        yield item

### Step 2: read cars only until the first 1975 model

In [17]:
early_models = take_until(
    pull_cars(DATA_FILE),
    lambda car: car.model_year >= 75,
)

for car in early_models:
    print(car.model_year, car.name)

70 Chevrolet Chevelle Malibu
70 Buick Skylark 320
70 Plymouth Satellite
70 AMC Rebel SST
70 Ford Torino
70 Ford Galaxie 500
70 Chevrolet Impala
70 Chevrolet Monte Carlo
71 Toyota Corolla
71 Datsun 510
70 Volkswagen 1131 Deluxe Sedan
70 Peugeot 504
70 Audi 100 LS
70 Saab 99e
70 BMW 2002
71 Chevrolet Vega 2300
71 Ford Pinto
70 AMC Gremlin
73 Mazda RX-3
72 Renault 12
74 Dodge Colt


Because the source is pulled lazily, rows after the first 1975 model are not requested by this pipeline.

# Problem 6 — Separate valid records from rejected records

A production pipeline often needs two outputs:

1. valid records that continue through the main pipeline;
2. rejected records that are saved for inspection.

We will call the rejected collection a **side channel**.

### Step 1: define a rejection record

In [18]:
@dataclass(frozen=True)
class RejectedRecord:
    raw: Dict[str, str]
    reason: str

### Step 2: create a parser that continues after errors

The rejected list is populated as the valid generator is consumed.

In [19]:
def parse_with_rejections(
    rows: Iterable[Dict[str, str]],
) -> Tuple[Iterator[CarRecord], List[RejectedRecord]]:
    rejected: List[RejectedRecord] = []

    def accepted() -> Iterator[CarRecord]:
        for row in rows:
            try:
                car = parse_record(row)
                if car.weight <= 0:
                    raise ValueError("weight must be positive")
                if car.cylinders <= 0:
                    raise ValueError("cylinders must be positive")
                yield car
            except (ValueError, KeyError) as exc:
                rejected.append(RejectedRecord(dict(row), str(exc)))

    return accepted(), rejected

### Step 3: test the side channel with deliberately bad rows

In [20]:
bad_rows = [
    {
        "name": "Valid Example",
        "mpg": "30",
        "cylinders": "4",
        "displacement": "100",
        "horsepower": "80",
        "weight": "2100",
        "acceleration": "15",
        "model_year": "80",
        "origin": "US",
    },
    {
        "name": "Broken Cylinders",
        "mpg": "25",
        "cylinders": "four",
        "displacement": "100",
        "horsepower": "80",
        "weight": "2200",
        "acceleration": "15",
        "model_year": "80",
        "origin": "US",
    },
    {
        "name": "Broken Weight",
        "mpg": "25",
        "cylinders": "4",
        "displacement": "100",
        "horsepower": "80",
        "weight": "-5",
        "acceleration": "15",
        "model_year": "80",
        "origin": "US",
    },
]

valid_stream, rejected_rows = parse_with_rejections(bad_rows)
valid_rows = list(valid_stream)

print("valid:", valid_rows)
print("rejected:")
for rejected in rejected_rows:
    print(" -", rejected.reason)

valid: [CarRecord(name='Valid Example', mpg=30.0, cylinders=4, displacement=100.0, horsepower=80.0, weight=2100.0, acceleration=15.0, model_year=80, origin='US')]
rejected:
 - invalid literal for int() with base 10: 'four'
 - weight must be positive


The side channel is empty until the valid stream is consumed. This is another consequence of lazy evaluation.

# Problem 7 — Impute missing horsepower with a running group average

The file contains a Ford Pinto with missing horsepower.

We will fill a missing value using the average horsepower already observed for the same origin.

This is a **stateful transformation** because the result depends on earlier records.

### Step 1: store running totals and counts

In [21]:
def impute_horsepower(
    cars: Iterable[CarRecord],
) -> Iterator[CarRecord]:
    horsepower_total: Dict[str, float] = defaultdict(float)
    horsepower_count: Dict[str, int] = defaultdict(int)

    for car in cars:
        if car.horsepower is not None:
            horsepower_total[car.origin] += car.horsepower
            horsepower_count[car.origin] += 1
            yield car
            continue

        count = horsepower_count[car.origin]
        if count == 0:
            yield car
        else:
            estimate = horsepower_total[car.origin] / count
            yield replace(car, horsepower=estimate)

### Step 2: inspect records around the missing value

In [22]:
for car in impute_horsepower(pull_cars(DATA_FILE)):
    if car.name == "Ford Pinto":
        print(car)

CarRecord(name='Ford Pinto', mpg=25.0, cylinders=4, displacement=98.0, horsepower=154.77777777777777, weight=2046.0, acceleration=19.0, model_year=71, origin='US')


### Important interpretation

This estimate uses only **previous** records. It is therefore:

- streaming;
- bounded in memory;
- sensitive to input order.

A two-pass global average would produce a different result but would require replaying or storing the source.

# Problem 8 — Remove duplicates while preserving order

The dataset contains a duplicated Honda Civic record.

Write a generator that keeps the first item for each key and drops later duplicates.

### Step 1: implement a generic `distinct_by` stage

In [23]:
def distinct_by(
    items: Iterable[T],
    key: Callable[[T], Any],
) -> Iterator[T]:
    seen = set()

    for item in items:
        marker = key(item)
        if marker in seen:
            continue
        seen.add(marker)
        yield item

### Step 2: deduplicate by the complete logical identity

For this tutorial, we define identity as `(name, model_year, origin)`.

In [24]:
all_cars = list(pull_cars(DATA_FILE))
unique_cars = list(
    distinct_by(
        pull_cars(DATA_FILE),
        key=lambda car: (car.name, car.model_year, car.origin),
    )
)

print("before:", len(all_cars))
print("after: ", len(unique_cars))

before: 31
after:  30


Exact duplicate detection requires remembering every key already seen. The state therefore grows with the number of distinct records.

# Problem 9 — Compute a rolling MPG average per origin

A rolling calculation needs a small amount of recent history.

For each origin, compute the average of the latest three non-missing MPG values.

### Step 1: use one bounded deque per origin

In [25]:
def rolling_mpg_by_origin(
    cars: Iterable[CarRecord],
    window_size: int,
) -> Iterator[Tuple[str, str, float]]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    windows: Dict[str, deque] = defaultdict(
        lambda: deque(maxlen=window_size)
    )

    for car in cars:
        if car.mpg is None:
            continue

        window = windows[car.origin]
        window.append(car.mpg)
        rolling_average = statistics.fmean(window)
        yield car.origin, car.name, rolling_average

### Step 2: inspect the first ten rolling results

In [26]:
rolling_results = rolling_mpg_by_origin(pull_cars(DATA_FILE), 3)

for origin, name, average in itertools.islice(rolling_results, 10):
    print(f"{origin:7} | {average:5.2f} | {name}")

US      | 18.00 | Chevrolet Chevelle Malibu
US      | 16.50 | Buick Skylark 320
US      | 17.00 | Plymouth Satellite
US      | 16.33 | AMC Rebel SST
US      | 17.00 | Ford Torino
US      | 16.00 | Ford Galaxie 500
US      | 15.33 | Chevrolet Impala
US      | 14.67 | Chevrolet Monte Carlo
Japan   | 31.00 | Toyota Corolla
Japan   | 29.00 | Datsun 510


Each deque stores at most three values, so memory for the windows is proportional to:

```text
number of origins × window size
```

# Problem 10 — Process the stream in bounded batches

Some downstream systems work more efficiently with batches.

Write a batching generator that never loads the entire input.

### Step 1: produce tuples of at most `batch_size` items

In [27]:
def in_batches(
    items: Iterable[T],
    batch_size: int,
) -> Iterator[Tuple[T, ...]]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    iterator = iter(items)

    while True:
        batch = tuple(itertools.islice(iterator, batch_size))
        if not batch:
            return
        yield batch

### Step 2: summarize each batch

In [28]:
def summarize_batch(batch: Sequence[CarRecord]) -> Dict[str, Any]:
    mpg_values = [car.mpg for car in batch if car.mpg is not None]

    return {
        "size": len(batch),
        "first_car": batch[0].name,
        "last_car": batch[-1].name,
        "average_mpg": statistics.fmean(mpg_values) if mpg_values else None,
    }

### Step 3: build the batching pipeline

In [29]:
batch_pipeline = transform(
    in_batches(pull_cars(DATA_FILE), 6),
    summarize_batch,
)

for summary in batch_pipeline:
    print(summary)

{'size': 6, 'first_car': 'Chevrolet Chevelle Malibu', 'last_car': 'Ford Galaxie 500', 'average_mpg': 16.5}
{'size': 6, 'first_car': 'Chevrolet Impala', 'last_car': 'Peugeot 504', 'average_mpg': 23.0}
{'size': 6, 'first_car': 'Audi 100 LS', 'last_car': 'AMC Gremlin', 'average_mpg': 24.833333333333332}
{'size': 6, 'first_car': 'Mazda RX-3', 'last_car': 'Chevrolet Monte Carlo S', 'average_mpg': 23.666666666666668}
{'size': 6, 'first_car': 'Chevrolet Monte Carlo Landau', 'last_car': 'Toyota Celica GT', 'average_mpg': 26.666666666666668}
{'size': 1, 'first_car': 'Honda Civic', 'last_car': 'Honda Civic', 'average_mpg': 33.0}


Only one batch is materialized at a time. After a summary is produced, the tuple can be released before the next batch is pulled.

# Problem 11 — Enrich records from a lookup table

A pipeline often combines streamed data with a small in-memory reference table.

Add a broad region label for each origin.

### Step 1: define the reference data and enriched record

In [30]:
REGION_BY_ORIGIN = {
    "US": "North America",
    "Japan": "Asia",
    "Europe": "Europe",
}


@dataclass(frozen=True)
class EnrichedCar:
    car: CarRecord
    region: str
    liters_per_100km: Optional[float]

### Step 2: convert MPG to liters per 100 km

The approximate conversion is:

```text
L/100 km = 235.214583 / MPG
```

In [31]:
def enrich_car(car: CarRecord) -> EnrichedCar:
    liters_per_100km = None
    if car.mpg not in {None, 0}:
        liters_per_100km = 235.214583 / car.mpg

    return EnrichedCar(
        car=car,
        region=REGION_BY_ORIGIN.get(car.origin, "Unknown"),
        liters_per_100km=liters_per_100km,
    )

### Step 3: inspect enriched records

In [32]:
for item in itertools.islice(
    transform(pull_cars(DATA_FILE), enrich_car),
    5,
):
    print(item.car.name, item.region, item.liters_per_100km)

Chevrolet Chevelle Malibu North America 13.067476833333334
Buick Skylark 320 North America 15.680972200000001
Plymouth Satellite North America 13.067476833333334
AMC Rebel SST North America 14.7009114375
Ford Torino North America 13.836151941176471


This lookup is inexpensive because the reference table is small and random access is constant-time on average.

# Problem 12 — Compute several group statistics in one pass

Suppose we need, for each origin:

- row count;
- count of known MPG values;
- average MPG;
- heaviest car.

A naive design might scan the file once for each statistic. We can compute all of them in one pass.

### Step 1: define mutable group state

In [33]:
@dataclass
class OriginSummaryState:
    count: int = 0
    mpg_count: int = 0
    mpg_total: float = 0.0
    heaviest: Optional[CarRecord] = None

    def update(self, car: CarRecord) -> None:
        self.count += 1

        if car.mpg is not None:
            self.mpg_count += 1
            self.mpg_total += car.mpg

        if self.heaviest is None or car.weight > self.heaviest.weight:
            self.heaviest = car

    def result(self) -> Dict[str, Any]:
        return {
            "count": self.count,
            "average_mpg": (
                self.mpg_total / self.mpg_count
                if self.mpg_count
                else None
            ),
            "heaviest_car": (
                self.heaviest.name if self.heaviest else None
            ),
            "heaviest_weight": (
                self.heaviest.weight if self.heaviest else None
            ),
        }

### Step 2: update one state object per origin

In [34]:
def summarize_origins(
    cars: Iterable[CarRecord],
) -> Dict[str, Dict[str, Any]]:
    states: Dict[str, OriginSummaryState] = defaultdict(OriginSummaryState)

    for car in cars:
        states[car.origin].update(car)

    return {
        origin: state.result()
        for origin, state in states.items()
    }

### Step 3: run the one-pass aggregation

In [35]:
origin_summaries = summarize_origins(pull_cars(DATA_FILE))

for origin, summary in sorted(origin_summaries.items()):
    print(origin, summary)

Europe {'count': 8, 'average_mpg': 25.5, 'heaviest_car': 'Mercedes-Benz 240d', 'heaviest_weight': 3250.0}
Japan {'count': 8, 'average_mpg': 29.6625, 'heaviest_car': 'Toyota Celica GT', 'heaviest_weight': 2665.0}
US {'count': 15, 'average_mpg': 18.646666666666665, 'heaviest_car': 'Chevrolet Impala', 'heaviest_weight': 4354.0}


The source is traversed once. The retained state is proportional to the number of groups, not the number of rows.

# Problem 13 — Keep the top three MPG cars per origin

Sorting every group stores all rows. A bounded heap stores only the current best *k* items per group.

### Step 1: maintain a min-heap for each origin

The smallest retained MPG stays at heap position zero. When a better car arrives, it replaces that item.

In [36]:
def top_k_mpg_per_origin(
    cars: Iterable[CarRecord],
    k: int,
) -> Dict[str, List[CarRecord]]:
    if k <= 0:
        raise ValueError("k must be positive")

    heaps: Dict[str, List[Tuple[float, int, CarRecord]]] = defaultdict(list)
    sequence = itertools.count()

    for car in cars:
        if car.mpg is None:
            continue

        entry = (car.mpg, next(sequence), car)
        heap = heaps[car.origin]

        if len(heap) < k:
            heapq.heappush(heap, entry)
        elif entry[0] > heap[0][0]:
            heapq.heapreplace(heap, entry)

    return {
        origin: [
            entry[2]
            for entry in sorted(heap, reverse=True)
        ]
        for origin, heap in heaps.items()
    }

### Step 2: inspect the result

In [37]:
top_by_origin = top_k_mpg_per_origin(pull_cars(DATA_FILE), 3)

for origin in sorted(top_by_origin):
    print(f"\n{origin}")
    for car in top_by_origin[origin]:
        print(f"  {car.mpg:4.1f} MPG | {car.name}")


Europe
  30.0 MPG | Mercedes-Benz 240d
  26.0 MPG | Renault 12
  26.0 MPG | BMW 2002

Japan
  33.8 MPG | Subaru GL
  33.0 MPG | Honda Civic
  33.0 MPG | Honda Civic

US
  28.0 MPG | Dodge Colt
  28.0 MPG | Chevrolet Vega 2300
  25.0 MPG | Ford Pinto


Memory is bounded by approximately:

```text
number of origins × k
```

This is much smaller than storing every row when `k` is small.

# Problem 14 — Understand iterator exhaustion and replayable sources

A generator object is single-pass. Once consumed, it is exhausted.

### Step 1: demonstrate exhaustion

In [38]:
one_time_stream = pull_cars(DATA_FILE)

first_count = sum(1 for _ in one_time_stream)
second_count = sum(1 for _ in one_time_stream)

print("first count: ", first_count)
print("second count:", second_count)

first count:  31
second count: 0


### Step 2: create a replayable iterable

An iterable object can create a fresh generator each time `iter(...)` is called.

In [39]:
class CarFile:
    def __init__(self, file_name: Path):
        self.file_name = file_name

    def __iter__(self) -> Iterator[CarRecord]:
        return pull_cars(self.file_name)

### Step 3: iterate twice

In [40]:
replayable = CarFile(DATA_FILE)

print(sum(1 for _ in replayable))
print(sum(1 for _ in replayable))

31
31


The data is not cached. Each traversal reopens and rereads the file. Replayability and caching are different design choices.

# Problem 15 — Build a safe configuration-driven pipeline

Suppose filters arrive from configuration rather than hard-coded Python.

Example:

```python
{"field": "mpg", "operator": "ge", "value": 28}
```

We should use an explicit operator registry, not `eval`.

### Step 1: define allowed operators

In [41]:
OPERATORS = {
    "eq": operator.eq,
    "ne": operator.ne,
    "gt": operator.gt,
    "ge": operator.ge,
    "lt": operator.lt,
    "le": operator.le,
}

### Step 2: compile one rule into a predicate

In [42]:
def compile_rule(rule: Dict[str, Any]) -> Callable[[CarRecord], bool]:
    field_name = rule["field"]
    operator_name = rule["operator"]
    expected = rule["value"]

    if operator_name not in OPERATORS:
        raise ValueError(f"Unsupported operator: {operator_name}")

    comparison = OPERATORS[operator_name]

    def predicate(car: CarRecord) -> bool:
        actual = getattr(car, field_name)
        if actual is None:
            return False
        return comparison(actual, expected)

    return predicate

### Step 3: combine several rules with logical AND

In [43]:
def compile_rules(
    rules: Sequence[Dict[str, Any]],
) -> Callable[[CarRecord], bool]:
    predicates = [compile_rule(rule) for rule in rules]
    return lambda car: all(predicate(car) for predicate in predicates)

### Step 4: run a configured query

In [44]:
rules = [
    {"field": "origin", "operator": "eq", "value": "Japan"},
    {"field": "mpg", "operator": "ge", "value": 30},
    {"field": "model_year", "operator": "ge", "value": 76},
]

configured_cars = where(
    pull_cars(DATA_FILE),
    compile_rules(rules),
)

for car in configured_cars:
    print(car)

CarRecord(name='Honda Civic', mpg=33.0, cylinders=4, displacement=91.0, horsepower=53.0, weight=1795.0, acceleration=17.4, model_year=76, origin='Japan')
CarRecord(name='Subaru GL', mpg=33.8, cylinders=4, displacement=97.0, horsepower=67.0, weight=2145.0, acceleration=18.0, model_year=80, origin='Japan')
CarRecord(name='Toyota Celica GT', mpg=32.0, cylinders=4, displacement=144.0, horsepower=96.0, weight=2665.0, acceleration=13.9, model_year=82, origin='Japan')
CarRecord(name='Honda Civic', mpg=33.0, cylinders=4, displacement=91.0, horsepower=53.0, weight=1795.0, acceleration=17.4, model_year=76, origin='Japan')


The operator allowlist prevents arbitrary code execution and makes supported behavior explicit.

# Problem 16 — Observe pull-based backpressure

In a pull pipeline, the consumer controls how quickly upstream stages run.

We can prove this by recording every source request.

### Step 1: instrument a source

In [45]:
def traced_source(limit: int, events: List[str]) -> Iterator[int]:
    events.append("source opened")
    try:
        for value in range(limit):
            events.append(f"source produced {value}")
            yield value
    finally:
        events.append("source closed")

### Step 2: request only three transformed even values

In [46]:
events: List[str] = []

result = list(
    itertools.islice(
        transform(
            where(
                traced_source(100, events),
                lambda value: value % 2 == 0,
            ),
            lambda value: value * value,
        ),
        3,
    )
)

print("result:", result)
print("events:")
for event in events:
    print(" -", event)

result: [0, 4, 16]
events:
 - source opened
 - source produced 0
 - source produced 1
 - source produced 2
 - source produced 3
 - source produced 4
 - source closed


The source produces `0` through `4` because the filter needs three even values. It does not produce all 100 values.

This demand propagation is the core behavior of a pull pipeline.

# Problem 17 — Write a lazy sink and return an audit record

A sink consumes a pipeline and creates an external effect, such as writing a file.

We will export efficient cars and return a compact audit summary.

### Step 1: define the audit record

In [47]:
@dataclass(frozen=True)
class ExportAudit:
    rows_written: int
    minimum_mpg: Optional[float]
    maximum_mpg: Optional[float]

### Step 2: implement the sink

The sink writes one row at a time and updates audit values as it goes.

In [48]:
def write_car_export(
    file_name: Path,
    cars: Iterable[CarRecord],
) -> ExportAudit:
    count = 0
    minimum_mpg = math.inf
    maximum_mpg = -math.inf

    with file_name.open("w", encoding="utf-8", newline="") as file_obj:
        writer = csv.DictWriter(
            file_obj,
            fieldnames=["name", "mpg", "model_year", "origin"],
        )
        writer.writeheader()

        for car in cars:
            writer.writerow(
                {
                    "name": car.name,
                    "mpg": car.mpg,
                    "model_year": car.model_year,
                    "origin": car.origin,
                }
            )
            count += 1

            if car.mpg is not None:
                minimum_mpg = min(minimum_mpg, car.mpg)
                maximum_mpg = max(maximum_mpg, car.mpg)

    return ExportAudit(
        rows_written=count,
        minimum_mpg=None if minimum_mpg == math.inf else minimum_mpg,
        maximum_mpg=None if maximum_mpg == -math.inf else maximum_mpg,
    )

### Step 3: export cars with at least 30 MPG

In [49]:
with tempfile.TemporaryDirectory() as temp_dir:
    export_file = Path(temp_dir) / "efficient_cars.csv"

    efficient_cars = where(
        pull_cars(DATA_FILE),
        lambda car: car.mpg is not None and car.mpg >= 30,
    )

    audit = write_car_export(export_file, efficient_cars)

    print(audit)
    print(export_file.read_text(encoding="utf-8"))

ExportAudit(rows_written=6, minimum_mpg=30.0, maximum_mpg=33.8)
name,mpg,model_year,origin
Toyota Corolla,31.0,71,Japan
Honda Civic,33.0,76,Japan
Mercedes-Benz 240d,30.0,80,Europe
Subaru GL,33.8,80,Japan
Toyota Celica GT,32.0,82,Japan
Honda Civic,33.0,76,Japan



A sink is terminal: after it consumes the iterator, the pipeline has no remaining values.

# Problem 18 — Capstone: build a complete reporting pipeline

Create a function that:

1. pulls typed records from the file;
2. removes duplicates;
3. imputes missing horsepower;
4. keeps models from 1975 onward;
5. keeps cars with MPG of at least 25;
6. computes a power-to-weight score;
7. keeps the best five scores;
8. returns serializable dictionaries.

We will build this in small steps.

### Step 1: define the report row

In [50]:
@dataclass(frozen=True)
class PerformanceRow:
    name: str
    origin: str
    model_year: int
    mpg: float
    horsepower: float
    weight: float
    power_to_weight: float

### Step 2: convert an eligible car into a report row

In [51]:
def to_performance_row(car: CarRecord) -> PerformanceRow:
    if car.mpg is None or car.horsepower is None:
        raise ValueError("MPG and horsepower are required")

    return PerformanceRow(
        name=car.name,
        origin=car.origin,
        model_year=car.model_year,
        mpg=car.mpg,
        horsepower=car.horsepower,
        weight=car.weight,
        power_to_weight=car.horsepower / car.weight,
    )

### Step 3: write the delegating pipeline function

In [52]:
def build_performance_report(file_name: Path) -> List[Dict[str, Any]]:
    cars: Iterable[CarRecord] = pull_cars(file_name)

    cars = distinct_by(
        cars,
        key=lambda car: (car.name, car.model_year, car.origin),
    )

    cars = impute_horsepower(cars)

    cars = where(
        cars,
        lambda car: (
            car.model_year >= 75
            and car.mpg is not None
            and car.mpg >= 25
            and car.horsepower is not None
        ),
    )

    performance_rows = transform(cars, to_performance_row)

    best_rows = heapq.nlargest(
        5,
        performance_rows,
        key=lambda row: row.power_to_weight,
    )

    return [asdict(row) for row in best_rows]

### Step 4: run the capstone solution

In [53]:
report = build_performance_report(DATA_FILE)

for row in report:
    print(row)

{'name': 'Toyota Celica GT', 'origin': 'Japan', 'model_year': 82, 'mpg': 32.0, 'horsepower': 96.0, 'weight': 2665.0, 'power_to_weight': 0.03602251407129456}
{'name': 'Honda Accord LX', 'origin': 'Japan', 'model_year': 78, 'mpg': 29.5, 'horsepower': 68.0, 'weight': 2135.0, 'power_to_weight': 0.03185011709601874}
{'name': 'Subaru GL', 'origin': 'Japan', 'model_year': 80, 'mpg': 33.8, 'horsepower': 67.0, 'weight': 2145.0, 'power_to_weight': 0.031235431235431235}
{'name': 'Honda Civic', 'origin': 'Japan', 'model_year': 76, 'mpg': 33.0, 'horsepower': 53.0, 'weight': 1795.0, 'power_to_weight': 0.029526462395543174}
{'name': 'Mercedes-Benz 240d', 'origin': 'Europe', 'model_year': 80, 'mpg': 30.0, 'horsepower': 67.0, 'weight': 3250.0, 'power_to_weight': 0.020615384615384615}


### Trace the final architecture

```text
report consumer
    <-- top 5 heap
    <-- performance mapping
    <-- eligibility filter
    <-- horsepower imputation
    <-- duplicate removal
    <-- typed parsing
    <-- normalized CSV rows
    <-- file
```

Most stages remain lazy. The final `heapq.nlargest` call is the point where the pipeline is fully consumed.

# Verification exercises

The next cells check important behavioral properties, not only example output.

## Test 1 — Projection does not consume more rows than requested

In [54]:
def test_projection_is_lazy() -> None:
    pulled: List[int] = []

    def source() -> Iterator[Dict[str, int]]:
        for value in range(10):
            pulled.append(value)
            yield {"value": value, "unused": value * 100}

    result = list(
        itertools.islice(
            select_fields(source(), "value"),
            3,
        )
    )

    assert result == [{"value": 0}, {"value": 1}, {"value": 2}]
    assert pulled == [0, 1, 2]


test_projection_is_lazy()
print("passed")

passed


## Test 2 — Duplicate removal preserves the first item

In [55]:
def test_distinct_preserves_first() -> None:
    values = [
        {"id": 1, "version": "first"},
        {"id": 2, "version": "only"},
        {"id": 1, "version": "second"},
    ]

    result = list(distinct_by(values, key=lambda item: item["id"]))

    assert result == [
        {"id": 1, "version": "first"},
        {"id": 2, "version": "only"},
    ]


test_distinct_preserves_first()
print("passed")

passed


## Test 3 — Batches handle an incomplete final group

In [56]:
def test_incomplete_batch() -> None:
    result = list(in_batches(range(8), 3))
    assert result == [(0, 1, 2), (3, 4, 5), (6, 7)]


test_incomplete_batch()
print("passed")

passed


## Test 4 — Replayable sources can be traversed repeatedly

In [57]:
def test_replayable_source() -> None:
    source = CarFile(DATA_FILE)
    first = sum(1 for _ in source)
    second = sum(1 for _ in source)
    assert first == second
    assert first > 0


test_replayable_source()
print("passed")

passed


# Further advanced problems

These extensions are intentionally left as practice after the complete solved tutorial.

## Extension A — Approximate duplicate detection

Replace the exact `seen` set with a Bloom filter. Discuss the false-positive tradeoff.

## Extension B — Merge sorted streams

Use `heapq.merge` to combine multiple files already sorted by model year without loading them all.

## Extension C — External sorting

Design a chunked external sort for a file too large to fit in memory.

## Extension D — Asynchronous pull source

Rebuild the source with an asynchronous generator and consume it with `async for`.

## Extension E — Retry a paginated API

Wrap transient failures with bounded exponential backoff while allowing validation errors to fail immediately.

## Extension F — Checkpointing

Store the last processed row number so a long pipeline can resume after interruption.

# Best-practice summary

When building pull pipelines:

1. normalize external schemas near the source;
2. convert strings to typed records early;
3. keep each generator responsible for one transformation;
4. make missing-data and rejection policies explicit;
5. use bounded state when possible;
6. document order-dependent stages;
7. remember that generator objects are single-pass;
8. distinguish replaying a source from caching its values;
9. avoid `eval` for configuration-driven behavior;
10. test laziness, early termination, state boundaries, and cleanup;
11. keep terminal sinks separate from reusable transformations;
12. materialize only where the algorithm truly requires it.

The central mental model remains:

```text
consumer <-- stage <-- stage <-- source
```

The consumer asks for the next value, and demand travels upstream one stage at a time.